# WSI – Ćwiczenie 5: Sztuczne Sieci Neuronowe

**Autorzy:** Mikołaj Wróbel, Kacper Maciejko  
**Data:** 2026  

---

## 1. Opis implementacji

### Architektura sieci (MLP)
- Implementacja oparta wyłącznie o **numpy** (bez PyTorch / TensorFlow).
- Sieć składa się z obiektów: `Model → [Layer] → [Node]`.
- Każdy `Node` przechowuje wektor wag i bias.
- Konfigurowalny: dowolna liczba warstw ukrytych i dowolna liczba neuronów.
- Funkcje aktywacji w warstwach ukrytych: **ReLU**.
- Warstwa wyjściowa: **Softmax**.
- Funkcja straty: **entropia krzyżowa** (cross-entropy).

### Algorytm optymalizacji
- **Mini-batch SGD** z propagacją wsteczną.
- Konfigurowalny współczynnik uczenia η i rozmiar mini-batcha.

### Zapis i wczytywanie modelu
- `model.save(path)` — zapisuje wagi, biasy i architekturę do pliku `.npz`.
- `Model.load(path)` — odtwarza gotowy model z pliku `.npz`.

### Normalizacja danych
- Piksele w zakresie 0–16 podzielone przez 16 → zakres **[0, 1]**.

### Podział danych (stratified)
- Trenujący: **70%**, Walidacyjny: **15%**, Testowy: **15%**.

### Augmentacja danych treningowych (×5)
Każdy obraz treningowy jest rozszerzany do 5 wersji:

| # | Technika | Opis |
|---|----------|------|
| 1 | Oryginał | bez zmian |
| 2 | Pixel dropout | losowe wyzerowanie 2 jasnych pikseli |
| 3 | Pogrubienie | maximum filter 2×2 |
| 4 | Przesunięcie | losowy shift o 1 piksel w osi X i/lub Y |
| 5 | Szum Gaussowski | σ = 1.5, wartości obcięte do [0, 16] |


In [3]:
# ── Setup ─────────────────────────────────────────────────────────────
from pathlib import Path
import sys
import numpy as np

ROOT_DIR = Path('').resolve().parents[0]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from Code.Import_training_data.import_traing_data import DataLoader
from Code.train.model import Model
from Code.train.training import TrainModel

loader = DataLoader()
X_train, y_train = loader.get_train_data()
X_val,   y_val   = loader.get_val_data()
X_test,  y_test  = loader.get_test_data()

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Classes: {np.unique(y_train)}')


Train: (6285, 64)  Val: (270, 64)  Test: (270, 64)
Classes: [0 1 2 3 4 5 6 7 8 9]


---
## 2. Eksperymenty porównawcze

Przetestowano **3 architektury × 2 współczynniki uczenia × 2 rozmiary mini-batcha**.

| # | Architektura | η | Batch | Test acc |
|---|---|---|---|---|
| 1 | 64→64→10 | 0.01 | pełny | 0.9556 |
| 2 | 64→128→10 | 0.01 | pełny | 0.9407 |
| 3 | 64→128→64→10 | 0.01 | pełny |0.9370 |
| 4 | 64→128→10 | 0.05 | pełny | 0.9593 |
| 5 | 64→128→10 | 0.01 | 32 | 0.9481 |
| 6 | 64→128→10 | 0.05 | 32 | 0.9630 |

In [4]:
# ── Helper: train one config and return (trainer, test_acc) ───────────
def run_experiment(layer_sizes, activations, lr, epochs, batch_size=None):
    model   = Model(layer_sizes, activations)
    trainer = TrainModel(model, learning_rate=lr)

    if batch_size is None:
        # Full-batch
        trainer.train(X_train, y_train, X_val, y_val, epochs=epochs, verbose=False)
    else:
        # Mini-batch SGD
        trainer.loss_history    = []
        trainer.val_acc_history = []
        n = X_train.shape[0]
        for epoch in range(1, epochs + 1):
            idx = np.random.permutation(n)
            for start in range(0, n, batch_size):
                batch = idx[start:start + batch_size]
                trainer.gradient_descent(X_train[batch], y_train[batch])
            if epoch % 5 == 0:
                loss    = trainer._cross_entropy_loss(model._forward(X_train), y_train)
                val_acc = model.accuracy(X_val, y_val)
                trainer.loss_history.append(round(float(loss), 5))
                trainer.val_acc_history.append(round(float(val_acc), 5))

    test_acc = model.accuracy(X_test, y_test)
    return trainer, test_acc


In [5]:
# ── Run all experiments ───────────────────────────────────────────────
EPOCHS = 200

experiments = [
    {'name': '64→64→10,        η=0.01, full',   'sizes': [64, 64, 10],       'act': ['relu','softmax'],        'lr': 0.01, 'batch': None},
    {'name': '64→128→10,       η=0.01, full',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.01, 'batch': None},
    {'name': '64→128→64→10,    η=0.01, full',   'sizes': [64, 128, 64, 10],  'act': ['relu','relu','softmax'], 'lr': 0.01, 'batch': None},
    {'name': '64→128→10,       η=0.05, full',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.05, 'batch': None},
    {'name': '64→128→10,       η=0.01, b=32',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.01, 'batch': 32},
    {'name': '64→128→10,       η=0.05, b=32',   'sizes': [64, 128, 10],      'act': ['relu','softmax'],        'lr': 0.05, 'batch': 32},
]

results = []
for exp in experiments:
    trainer, test_acc = run_experiment(
        exp['sizes'], exp['act'], exp['lr'], EPOCHS, exp['batch']
    )
    exp['trainer']  = trainer
    exp['test_acc'] = test_acc
    results.append(exp)
    print(f"{exp['name']:<40}  test acc = {test_acc:.4f}")

64→64→10,        η=0.01, full             test acc = 0.9667
64→128→10,       η=0.01, full             test acc = 0.9704
64→128→64→10,    η=0.01, full             test acc = 0.9556
64→128→10,       η=0.05, full             test acc = 0.9667
64→128→10,       η=0.01, b=32             test acc = 0.9630
64→128→10,       η=0.05, b=32             test acc = 0.9704


---
## 3. Krzywe uczenia

In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Training Loss', 'Validation Accuracy'),
)

colors = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']

for i, exp in enumerate(results):
    t = exp['trainer']
    xs = [j * 5 for j in range(1, len(t.loss_history) + 1)]
    c  = colors[i % len(colors)]

    fig.add_trace(go.Scatter(x=xs, y=t.loss_history,
                             name=exp['name'], line=dict(color=c),
                             legendgroup=exp['name']),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=xs, y=t.val_acc_history,
                             name=exp['name'], line=dict(color=c),
                             legendgroup=exp['name'], showlegend=False),
                  row=1, col=2)

fig.update_xaxes(title_text='Epoch', gridcolor='#333')
fig.update_yaxes(gridcolor='#333')
fig.update_layout(
    title='Learning Curves – all experiments',
    plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
    font=dict(color='white'), height=450,
)
fig.show()

---
## 3c. Optymalizacja Bayesowska

Zamiast ręcznie dobierać hiperparametry, używamy **Bayesian Optimisation** z procesem Gaussa (GP) jako modelem zastępczym i funkcją akwizycji UCB.

### Przeszukiwana przestrzeń

| Parametr               | Zakres      |
|------------------------|-------------|
| Liczba warstw ukrytych | 1 – 4       |
| Neurony na warstwę     | 16 – 128    |
| Współczynnik uczenia η | 0.001 – 0.1 |
| Liczba epok            | 50 – 200    |
| Rozmiar mini-batcha    | 16 – 128    |

### Procedura
1. **Faza losowa** — `n_random_starts` konfiguracji losowanych jednostajnie (eksploracja).
2. **Faza GP** — GP dopasowuje się do wyników; UCB (`mean + κ·std`) wskazuje następny kandydat.
3. Po wyczerpaniu budżetu iteracji zwracana jest konfiguracja z najwyższą val accuracy.
4. **Re-trening ×5** — model jest trenowany 5 razy z tą konfiguracją; zachowywany jest ten z najwyższą **val accuracy** (unikamy data leakage przez test set).
5. Opcjonalnie model jest zapisywany do `saved_models/bayes_best_model.npz`.


In [7]:
from Code.Optymalize_Model_Size.Bayes_optymalization import BayesOptimizer

# Run Bayesian optimisation over the architecture / hyperparameter space
optimizer = BayesOptimizer(
    X_train, y_train,
    X_val,   y_val,
    layers_range    = (1, 4),
    nodes_range     = (16, 128),
    lr_range        = (1e-3, 0.1),
    epochs_range    = (50, 200),
    n_random_starts = 3,
    n_iterations    = 30,
)
bayes_config, bayes_val_acc, bayes_trainer = optimizer.optimise()

# Reconstruct layer sizes and activation list from the found config
_input_size  = X_train.shape[1]   # 64
_output_size = 10

bayes_hidden = [bayes_config[f'nodes_{i+1}'] for i in range(bayes_config['num_layers'])]
bayes_sizes = [_input_size] + bayes_hidden + [_output_size]
bayes_acts = ['relu'] * bayes_config['num_layers'] + ['softmax']

# Evaluate on held-out test set (model is already trained — no re-training)
bayes_test_acc = bayes_trainer.model.accuracy(X_test, y_test)
print(f"\nBayes best → layers={bayes_config['num_layers']}, "
      f"nodes={bayes_hidden}, "
      f"lr={bayes_config['lr']:.5f}, epochs={bayes_config['epochs']}")
print(f"Val accuracy:  {bayes_val_acc:.4f}")
print(f"Test accuracy: {bayes_test_acc:.4f}")

# Build a 'best' dict compatible with the visualization cell below
best = {
    'name'     : (f"Bayes: {bayes_config['num_layers']}×{bayes_hidden} "
                  f"lr={bayes_config['lr']:.4f} ep={bayes_config['epochs']}"),
    'sizes'    : bayes_sizes,
    'act'      : bayes_acts,
    'trainer'  : bayes_trainer,
    'test_acc' : bayes_test_acc,
}


  Bayesian Optimisation - searching for best architecture and batch size

[ 1/30] random | layers=2, nodes=[116, 22], lr=0.0275, epochs=59, batch_size=46
  → Val accuracy: 0.7889

[ 2/30] random | layers=1, nodes=[57], lr=0.0860, epochs=128, batch_size=101
  → Val accuracy: 0.9593

[ 3/30] random | layers=2, nodes=[62, 35], lr=0.0910, epochs=108, batch_size=35
  → Val accuracy: 0.9593

[ 4/30] GP-guided | layers=2, nodes=[94, 117], lr=0.0042, epochs=167, batch_size=48
  → Val accuracy: 0.9519

[ 5/30] GP-guided | layers=1, nodes=[62], lr=0.0640, epochs=183, batch_size=122
  → Val accuracy: 0.9667

[ 6/30] GP-guided | layers=3, nodes=[16, 104, 24], lr=0.0436, epochs=198, batch_size=71
  → Val accuracy: 0.6370

[ 7/30] GP-guided | layers=3, nodes=[76, 111, 104], lr=0.0629, epochs=85, batch_size=116
  → Val accuracy: 0.1000

[ 8/30] GP-guided | layers=1, nodes=[29], lr=0.0074, epochs=198, batch_size=23
  → Val accuracy: 0.9444

[ 9/30] GP-guided | layers=1, nodes=[101], lr=0.0947, epochs=

In [8]:
# # ── Export Bayes-optimised model to disk ─────────────────────────────
# from pathlib import Path

# MODELS_DIR = Path('saved_models')
# MODELS_DIR.mkdir(exist_ok=True)

# save_path = MODELS_DIR / 'bayes_best_model'
# bayes_trainer.model.save(str(save_path))
# print(f"Model saved to {save_path}.npz")
# print(f"  Architecture : {bayes_sizes}")
# print(f"  Activations  : {bayes_acts}")
# print(f"  Val accuracy : {bayes_val_acc:.4f}")
# print(f"  Test accuracy: {bayes_test_acc:.4f}")


---
## 3b. Wizualizacje interaktywne

Poniższe wykresy otwierają się w przeglądarce:
- **Architektura 3D** – obracaj myszą, przewijaj aby przybliżyć.
- **Postęp trenowania** – najedź kursorem na punkt aby zobaczyć dokładne wartości.


In [9]:
from Code.Visualization.network_visualization import show_architecture_3d, show_training_progress

# 'best' is set by the Bayesian optimisation cell above

# ── 3D architecture of the best experiment ────────────────────────────
best_sizes = best['sizes']
best_acts  = best['act']

print(f"Showing 3D architecture for: {best['name']}")
show_architecture_3d(best_sizes, best_acts)

# ── Animated training progress of the best experiment ─────────────────
print(f"Showing training progress for: {best['name']}")
show_training_progress(best['trainer'].loss_history, best['trainer'].val_acc_history)


Showing 3D architecture for: Bayes: 1×[62] lr=0.0640 ep=183


Showing training progress for: Bayes: 1×[62] lr=0.0640 ep=183


In [10]:
from Code.Visualization.network_visualization import show_architecture

fig_all = go.Figure()
colors  = ['#636EFA','#EF553B','#00CC96','#AB63FA','#FFA15A','#19D3F3']

for i, exp in enumerate(results):
    t  = exp['trainer']
    xs = [j * 5 for j in range(1, len(t.val_acc_history) + 1)]
    c  = colors[i % len(colors)]

    fig_all.add_trace(go.Scatter(
        x=xs, y=t.val_acc_history,
        mode='lines+markers',
        name=exp['name'],
        line=dict(color=c, width=2),
        marker=dict(size=5),
        hovertemplate='Epoch %{x}<br>Val Acc: %{y:.4f}<extra></extra>',
    ))

fig_all.update_layout(
    title='Validation Accuracy – all experiments',
    xaxis_title='Epoch',
    yaxis_title='Val Accuracy',
    yaxis=dict(range=[0, 1.05], gridcolor='#333'),
    xaxis=dict(gridcolor='#333'),
    plot_bgcolor='#1e1e2e',
    paper_bgcolor='#1e1e2e',
    font=dict(color='white'),
    height=420,
    legend=dict(font=dict(size=10)),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=1.15, x=0.5, xanchor='center',
        buttons=[
            dict(
                label='▶  Play',
                method='animate',
                args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True)],
            ),
            dict(
                label='⏸  Pause',
                method='animate',
                args=[[None], dict(frame=dict(duration=0), mode='immediate')],
            ),
        ],
    )],
)

# Build animation frames – each frame reveals one more epoch point
max_points = max(len(e['trainer'].val_acc_history) for e in results)
frames = []
for frame_idx in range(1, max_points + 1):
    frame_traces = []
    for i, exp in enumerate(results):
        t  = exp['trainer']
        xs = [j * 5 for j in range(1, len(t.val_acc_history) + 1)]
        visible_x = xs[:frame_idx]
        visible_y = t.val_acc_history[:frame_idx]
        frame_traces.append(go.Scatter(x=visible_x, y=visible_y))
    frames.append(go.Frame(data=frame_traces, name=str(frame_idx)))

fig_all.frames = frames
fig_all.show()


---
## 4. Najlepsza konfiguracja – macierz pomyłek

In [11]:
# Pick the best experiment by test accuracy
best = max(results, key=lambda e: e['test_acc'])
print(f"Best config : {best['name']}")
print(f"Test accuracy: {best['test_acc']:.4f}")

# Confusion matrix
best_trainer = best['trainer']
y_pred_proba = best_trainer.model._forward(X_test)
y_pred       = np.argmax(y_pred_proba, axis=1)

n_classes = 10
cm = np.zeros((n_classes, n_classes), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true][pred] += 1

fig_cm = go.Figure(go.Heatmap(
    z=cm,
    x=[str(i) for i in range(n_classes)],
    y=[str(i) for i in range(n_classes)],
    colorscale='Blues',
    text=cm, texttemplate='%{text}',
    hovertemplate='True: %{y}<br>Pred: %{x}<br>Count: %{z}<extra></extra>',
))
fig_cm.update_layout(
    title=f'Confusion Matrix – {best["name"]}',
    xaxis_title='Predicted', yaxis_title='True',
    plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
    font=dict(color='white'), height=500,
    yaxis=dict(autorange='reversed'),
)
fig_cm.show()


Best config : 64→128→10,       η=0.01, full
Test accuracy: 0.9704


---
## 5. Wyniki w formie tabeli

In [12]:
import pandas as pd

df = pd.DataFrame([
    {
        'Konfiguracja': e['name'],
        'η':            e['lr'],
        'Batch':        e['batch'] if e['batch'] else 'full',
        'Test Acc':     f"{e['test_acc']:.4f}",
        'Best Val Acc': f"{max(e['trainer'].val_acc_history):.4f}" if e['trainer'].val_acc_history else 'N/A',
    }
    for e in results
])
df

,Konfiguracja,η,Batch,Test Acc,Best Val Acc
0,"64→64→10, η=0.01, full",0.01,full,0.9667,0.9444
1,"64→128→10, η=0.01, full",0.01,full,0.9704,0.9556
2,"64→128→64→10, η=0.01, full",0.01,full,0.9556,0.9556
3,"64→128→10, η=0.05, full",0.05,full,0.9667,0.9704
4,"64→128→10, η=0.01, b=32",0.01,32,0.9630,0.9593
5,"64→128→10, η=0.05, b=32",0.05,32,0.9704,0.9741


## 5. Wnioski i podsumowanie eksperymentów

Na podstawie przeprowadzonych badań, w tym analizy manualnej oraz 30 iteracji automatycznej Optymalizacji Bayesowskiej, sformułowano następujące wnioski merytoryczne:

### 1. Wpływ architektury (liczba warstw i neuronów)
* **Problem głębokich sieci (Zjawisko zanikania gradientu / Przeuczenie):** Zwiększanie liczby warstw drastycznie utrudniało proces uczenia. W iteracji nr 4 (3 warstwy ukryte: `[99, 44, 42]`) sieć osiągnęła walidacyjną dokładność na poziomie zaledwie **41.48%**, a w iteracji nr 6 (`[54, 98, 57]`) model kompletnie utknął w minimum lokalnym ze skutecznością **10.00%**.
* **Efektywność płytkich struktur:** Najwyższą stabilność i powtarzalność wyników wykazały sieci z **1 warstwą ukrytą**.

### 2. Wpływ współczynnika uczenia ($\eta$) 
* **Agresywny współczynnik uczenia:** W przypadku struktur jednowarstwowych, sieć tolerowała wyższe wartości parametru.
* **Wrażliwość w sieciach wielowarstwowych:** Przy sieciach z 2 warstwami ukrytymi, wysoki $\eta$ uniemożliwiał zbieżność. Dopiero obniżenie współczynnika uczenia przywróciło stabilność optymalizacji i pozwoliło dwuwarstwowej sieci osiągnąć wysoką skuteczność na poziomie **93.33%**.

### 3. Wpływ rozmiaru mini-batcha
* **Mniejsze paczki (batch_size 23 - 40):** Zapewniały częstszą aktualizację wag wewnątrz jednej epoki. Stochastyczny szum wprowadzany przez mniejsze podzbiory danych skutecznie pomagał modelom jednowarstwowym oraz dobrze nastrojonym dwuwarstwowym omijać lokalne minima i szybciej uzyskiwać wysokie wskaźniki dokładności (Val > 93%).
* **Większe paczki (batch_size full):** Pogłębiały problemy ze zbieżnością, spowalniając proces korekty wag.

### 4. Najlepsza konfiguracja (Wynik Optymalizacji Bayesowskiej)
* Proces optymalizacji bayesowskiej (GP-guided) zidentyfikował optymalne parametry:
  * Liczba warstw ukrytych: **1**
  * Liczba neuronów: **28**
  * Współczynnik uczenia ($\eta$): **0.06212**
  * Liczba epok: **129**
* Model ten osiągnął dokładność **97.78% na zbiorze walidacyjnym** oraz ostateczny wynik **96.67% na zbiorze testowym**.

### 5. Obserwacje z macierzy pomyłek (Confusion Matrix)
* **Typowe pomyłki:** Najczęstsze pomyłki zachodzą dla **8** . 

---
## 7. Interaktywny podgląd predykcji

Wybierz cyfrę z dropdown i kliknij **▶ Losuj próbkę**, aby zobaczyć prawdziwy przykład z zestawu testowego i predykcję modelu.


In [13]:
import ipywidgets as widgets
from IPython.display import display
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Controls ─────────────────────────────────────────────────────────
_digit_dd = widgets.Dropdown(
    options=[('All', -1)] + [(str(i), i) for i in range(10)],
    value=-1,
    description='Cyfra:',
    layout=widgets.Layout(width='130px'),
)
_sample_btn = widgets.Button(
    description='▶ Losuj próbkę',
    button_style='primary',
    layout=widgets.Layout(width='150px'),
)
_out = widgets.Output()

# ── Callback ─────────────────────────────────────────────────────────
def _on_sample(_):
    digit = _digit_dd.value
    if digit == -1:
        idx = np.random.randint(len(X_test))
    else:
        indices = np.where(y_test == digit)[0]
        idx = np.random.choice(indices)

    x     = X_test[idx].reshape(1, -1)
    pr    = best['trainer'].model._forward(x)[0]
    p     = int(np.argmax(pr))
    true  = int(y_test[idx])
    img   = X_test[idx].reshape(8, 8)

    correct = p == true
    color   = '#00CC96' if correct else '#EF553B'
    verdict = '✓ Poprawnie' if correct else f'✗ Błąd (prawdziwa: {true})'

    with _out:
        _out.clear_output(wait=True)

        fig = make_subplots(
            rows=1, cols=2,
            column_widths=[0.35, 0.65],
            subplot_titles=(f'Wejście (true={true})', f'Predykcja: {p}  {verdict}'),
        )

        # Left: digit image as heatmap
        fig.add_trace(go.Heatmap(
            z=img[::-1],
            colorscale='gray',
            showscale=False,
            hovertemplate='Wiersz %{y}, Kolumna %{x}<br>Wartość: %{z}<extra></extra>',
        ), row=1, col=1)

        # Right: probability bar chart
        fig.add_trace(go.Bar(
            x=list(range(10)),
            y=pr.tolist(),
            text=[f'{v:.0%}' for v in pr],
            textposition='outside',
            marker_color=['#00CC96' if i == p else '#636EFA' for i in range(10)],
            hovertemplate='Cyfra %{x}: %{y:.4f}<extra></extra>',
        ), row=1, col=2)

        fig.update_layout(
            height=320,
            plot_bgcolor='#1e1e2e', paper_bgcolor='#1e1e2e',
            font=dict(color='white'),
            margin=dict(t=50, b=30),
            showlegend=False,
        )
        fig.update_xaxes(gridcolor='#333', row=1, col=2,
                         tickmode='array', tickvals=list(range(10)))
        fig.update_yaxes(gridcolor='#333', range=[0, 1.2], row=1, col=2)
        fig.update_xaxes(showticklabels=False, row=1, col=1)
        fig.update_yaxes(showticklabels=False, row=1, col=1)
        fig.update_traces(
            selector=dict(type='heatmap'),
            colorbar=dict(len=0.5),
        )
        fig.show()

_sample_btn.on_click(_on_sample)
display(widgets.HBox([_digit_dd, _sample_btn]))
display(_out)

# Show one sample immediately on load
_on_sample(None)


Output()